Step 1: Imports & Directory Setup

In [2]:
import json
from pathlib import Path
import pandas as pd
import requests

BASE_DIR = Path("..")
BRONZE_WEATHER_DIR = BASE_DIR / "bronze" / "weather"
SILVER_DIR = BASE_DIR / "silver"

BRONZE_WEATHER_DIR.mkdir(parents=True, exist_ok=True)
SILVER_DIR.mkdir(parents=True, exist_ok=True)

Step 2: Determine Date Range from Existing Hive Data

In [3]:
# Read existing hive data from the Silver layer to align timeframes
df_hives = pd.read_parquet(SILVER_DIR / "weight_silver.parquet")

# Ensure the timestamp column is in datetime format
# Adjust 'timestamp' to match your actual column name if needed
date_column = "timestamp" if "timestamp" in df_hives.columns else df_hives.columns[0]
df_hives[date_column] = pd.to_datetime(df_hives[date_column])

# Extract start and end dates
start_date = df_hives[date_column].dt.date.min().strftime("%Y-%m-%d")
end_date = df_hives[date_column].dt.date.max().strftime("%Y-%m-%d")

print(f"🗓️ Hive observation date range: {start_date} to {end_date}")

🗓️ Hive observation date range: 2017-01-01 to 2019-05-31


Step 3: Fetch & Save Raw Weather Data (Bronze Layer)

In [4]:
# Bad Schwartau location coordinates
LATITUDE = 53.92
LONGITUDE = 10.70


def fetch_and_save_raw_weather(
    lat: float, lon: float, date_from: str, date_to: str, output_dir: Path
) -> dict | None:
    """Fetches raw weather data from the BrightSky API and saves the unedited JSON to the Bronze layer."""
    url = f"https://api.brightsky.dev/weather?lat={lat}&lon={lon}&date={date_from}&last_date={date_to}"

    print(f"📡 Requesting weather data from BrightSky API ({date_from} to {date_to})...")

    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        raw_data = response.json()

        # Define raw file destination
        file_path = output_dir / f"weather_raw_{date_from}_to_{date_to}.json"

        # Save immutable raw JSON response (Bronze Layer)
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(raw_data, f, ensure_ascii=False, indent=4)

        print(f"✅ Raw weather data successfully saved to: {file_path}")
        return raw_data

    except requests.exceptions.RequestException as e:
        print(f"❌ API Request failed: {e}")
        return None


# Execute data retrieval
raw_weather_data = fetch_and_save_raw_weather(
    LATITUDE, LONGITUDE, start_date, end_date, BRONZE_WEATHER_DIR
)

📡 Requesting weather data from BrightSky API (2017-01-01 to 2019-05-31)...
✅ Raw weather data successfully saved to: ../bronze/weather/weather_raw_2017-01-01_to_2019-05-31.json


Step 4: Parse & Clean JSON Data

In [5]:
# Extract weather observations list from the response
weather_records = raw_weather_data.get("weather", [])

# Convert JSON array to DataFrame
df_weather_raw = pd.DataFrame(weather_records)

print(f"📊 Extracted {len(df_weather_raw)} hourly weather records.")
df_weather_raw.head()

📊 Extracted 21121 hourly weather records.


,timestamp,source_id,precipitation,pressure_msl,sunshine,temperature,wind_direction,wind_speed,cloud_cover,dew_point,relative_humidity,visibility,wind_gust_direction,wind_gust_speed,condition,precipitation_probability,precipitation_probability_6h,solar,fallback_source_ids,icon
0,2017-01-01T00:00:00+00:00,286541,0.0,1020.3,NaN,3.9,220,8.6,87.0,3.2,95,2260,210,18.7,dry,None,None,0.0,"{'temperature': 7102, 'wind_gust_speed': 7102,...",cloudy
1,2017-01-01T01:00:00+00:00,286541,0.0,1019.6,NaN,4.1,230,9.0,100.0,3.3,94,940,240,22.3,dry,None,None,0.0,"{'temperature': 7102, 'wind_gust_speed': 7102,...",cloudy
2,2017-01-01T02:00:00+00:00,286541,0.0,1019.1,NaN,4.1,230,8.6,100.0,3.1,93,1740,230,20.5,dry,None,None,0.0,"{'temperature': 7102, 'wind_gust_speed': 7102,...",cloudy
3,2017-01-01T03:00:00+00:00,286541,0.0,1018.1,0.0,4.1,220,7.6,100.0,3.0,93,3230,210,19.8,dry,None,None,0.0,"{'temperature': 7102, 'wind_gust_speed': 7102,...",cloudy
4,2017-01-01T04:00:00+00:00,286541,0.0,1017.2,0.0,4.2,230,7.9,100.0,3.1,92,2660,250,22.3,dry,None,None,0.0,"{'temperature': 7102, 'wind_gust_speed': 7102,...",cloudy


Step 5: Transform & Standardize (Silver Layer Requirements)

In [6]:
def process_weather_to_silver(df: pd.DataFrame) -> pd.DataFrame:
    """Cleans, formats, and selects relevant weather attributes for analysis."""
    # Create a explicit copy to avoid slice warnings
    df_clean = df.copy()

    # 1. Convert timestamp to datetime
    df_clean["timestamp"] = pd.to_datetime(df_clean["timestamp"])

    # 2. Select key columns relevant to hive activity
    # BrightSky fields: timestamp, temperature, relative_humidity, precipitation, wind_speed
    columns_to_keep = [
        "timestamp",
        "temperature",
        "relative_humidity",
        "precipitation",
        "wind_speed",
    ]

    # Filter only available target columns
    available_cols = [col for col in columns_to_keep if col in df_clean.columns]
    df_clean = df_clean[available_cols]

    # 3. Rename columns for clarity and consistency
    rename_mapping = {
        "relative_humidity": "humidity",
        "precipitation": "precipitation_mm",
        "wind_speed": "wind_speed_kmh",
    }
    df_clean = df_clean.rename(columns=rename_mapping)

    # 4. Handle missing values (e.g., forward-fill brief missing sensor readings)
    df_clean = df_clean.ffill().bfill()

    # 5. Sort by timestamp
    df_clean = df_clean.sort_values(by="timestamp").reset_index(drop=True)

    return df_clean


# Transform data
df_weather_silver = process_weather_to_silver(df_weather_raw)
print("✅ Weather data cleaned and standardized.")
df_weather_silver.info()

✅ Weather data cleaned and standardized.
<class 'pandas.DataFrame'>
RangeIndex: 21121 entries, 0 to 21120
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   timestamp         21121 non-null  datetime64[us, UTC]
 1   temperature       21121 non-null  float64            
 2   humidity          21121 non-null  int64              
 3   precipitation_mm  21121 non-null  float64            
 4   wind_speed_kmh    21121 non-null  float64            
dtypes: datetime64[us, UTC](1), float64(3), int64(1)
memory usage: 825.2 KB


Step 6: Save to Silver Layer as Parquet

In [7]:
# Define Silver Parquet path
silver_parquet_path = SILVER_DIR / "weather_silver.parquet"

# Save to Parquet format with compression for efficiency
df_weather_silver.to_parquet(silver_parquet_path, index=False)

print(f"🎉 Silver weather file successfully saved to: {silver_parquet_path}")

🎉 Silver weather file successfully saved to: ../silver/weather_silver.parquet
